In [2]:
from dataclasses import dataclass
import tiktoken


@dataclass
class TokenResult:
    model_name: str
    tokens: list[str]
    token_ids: list[int]
    token_count: int


def _tokenize_with_tiktoken(text: str, model: str) -> TokenResult:
    enc = tiktoken.encoding_for_model(model)

    token_ids = enc.encode(text)
    tokens = enc.decode_tokens_bytes(token_ids)

    # print(tokens)
    # print(token_ids)
    # print(model)
    # print(len(token_ids))

    return TokenResult(
        model_name=model,
        tokens=tokens,
        token_ids=token_ids,
        token_count=len(token_ids),
    )


encoder = tiktoken.encoding_for_model("gpt-4o")  # GPT-4 tokenizer


def tokenize(text: str, model: str) -> TokenResult:
    model_aliases = {
        "gpt-3": "gpt-3.5-turbo",
    }
    resolved_model = model_aliases.get(model, model)

    if resolved_model in {"gpt-4", "gpt-3.5-turbo", "gpt-4o"}:
        return _tokenize_with_tiktoken(text, resolved_model)

    raise ValueError(f"Unsupported model: {model}")


result = tokenize("The capital of France is Paris.", "gpt-4")
print(result)

result2 = tokenize("My Name is Kamran Javed", "gpt-3")
print(result2)


TokenResult(model_name='gpt-4', tokens=[b'The', b' capital', b' of', b' France', b' is', b' Paris', b'.'], token_ids=[791, 6864, 315, 9822, 374, 12366, 13], token_count=7)
TokenResult(model_name='gpt-3.5-turbo', tokens=[b'My', b' Name', b' is', b' Kam', b'ran', b' J', b'aved'], token_ids=[5159, 4076, 374, 29549, 6713, 622, 4234], token_count=7)


In [ ]:
from tokenizer_explorer.tokenizers import tokenize

type MultiLangDict = dict[str, dict[str, str]]


MULTILINGUAL_SAMPLES: MultiLangDict = {
    "greeting": {
        "en": "Hello, how are you?",
        "ja": "こんにちは、お元気ですか？",
        "ar": "مرحبا، كيف حالك؟",
        "ko": "안녕하세요, 어떻게 지내세요?",
        "zh": "你好，你怎么样？",
        "de": "Hallo, wie geht es Ihnen?",
        "hi": "नमस्ते, आप कैसे हैं?",
        "ru": "Здравствуйте, как дела?",
    },
    "code": {
        "python": "def hello():\n    return 'world'",
        "javascript": "const hello = () => 'world';",
        "sql": "SELECT name FROM users WHERE active = true;",
    },
}


def explore_multilingual(samples: MultiLangDict, model: str = "gpt-4o") -> None:
    """
    For each concept in the sample dictionary, tokenize every language/variant
    and print the token count, ratio vs English baseline, and raw token IDs.
    """

    for concept, variants in samples.items():
        print(f"\n{'=' * 60}")
        print(f"CONCEPT: {concept.upper()}")
        print(f"{'=' * 60}")

        # Get English (or first key) as baseline
        baseline_key = "en" if "en" in variants else next(iter(variants))
        baseline = tokenize(variants[baseline_key], model)
        print(f"[baseline: {baseline_key}], [baseline_count: {baseline.token_count}]")

        for lang, text in variants.items():
            result = tokenize(text, model)
            ratio = result.token_count / baseline.token_count
            print(
                f"  {lang:>12}  |  tokens: {result.token_count:>3}  |  ratio: {ratio:.2f}x  |  {text[:40]}"
            )
            print(
                f"              |  IDs: {result.token_ids[:8]}{'...' if len(result.token_ids) > 8 else ''}"
            )


result2 = explore_multilingual(MULTILINGUAL_SAMPLES)
print(result2)


In [2]:
import os

import numpy as np
from dotenv import load_dotenv
from openai import OpenAI


# reads .env file into os.environ
load_dotenv()

# automatically reads OPENAI_API_KEY from environment
client = OpenAI()


SIMILARITY_SAMPLES: list[tuple[str, str]] = [
    # label, text
    ("dog_1", "The dog ran across the park"),
    ("dog_2", "A puppy sprinted through the field"),
    ("weather", "It is raining heavily outside today"),
    ("code", "def add(a, b): return a + b"),
]


def get_embeddings(texts: list[str], model: str = "text-embedding-3-small") -> np.ndarray:
    response = client.embeddings.create(input=texts, model=model)
    vectors = [item.embedding for item in response.data]
    return np.array(vectors, dtype=np.float32)


texts = [text for _, text in SIMILARITY_SAMPLES]
result = get_embeddings(texts)

# print(result)


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """
    Cosine similarity between two pre-normalized vectors.
    Because OpenAI embeddings are unit vectors and comes normalized, this is just the dot product.
    Score range: 0.0 (unrelated) → 1.0 (identical meaning).
    """
    return float(np.dot(a, b))


def explore_similarity(samples: list[tuple[str, str]]) -> None:

    labels = [label for label, _ in samples]
    texts = [text for _, text in samples]
    print(f"LABELS: {labels}")

    # Step 1: Embed all texts at once
    embeddings = get_embeddings(texts)
    print(f"\nEmbedding shape: {embeddings.shape}")
    print(
        f"  → {embeddings.shape[0]} sentences, each a {embeddings.shape[1]}-dimensional vector\n"
    )

    # Step 2: Build a simple index so we can look up by label
    idx = {label: i for i, label in enumerate(labels)}
    print(f"IDX: {idx}")

    # Step 3: Compare specific pairs
    pairs_to_compare = [
        ("dog_1", "dog_2"),  # Should be HIGH — same concept, different words
        ("dog_1", "weather"),  # Should be LOW — unrelated topics
        ("dog_1", "code"),  # Should be VERY LOW — completely different domains
        ("dog_2", "weather"),  # Middle ground
    ]

    print("Pairwise cosine similarity:")
    print(f"  {'Pair':<30}  Score")
    print(f"  {'-' * 30}  -----")

    for label_a, label_b in pairs_to_compare:
        similarity = cosine_similarity(embeddings[idx[label_a]], embeddings[idx[label_b]])
        print(f" {label_a} <--> {label_b:<18}  {similarity:.4f}")


result2 = explore_similarity(SIMILARITY_SAMPLES)

print(result2)


LABELS: ['dog_1', 'dog_2', 'weather', 'code']

Embedding shape: (4, 1536)
  → 4 sentences, each a 1536-dimensional vector

IDX: {'dog_1': 0, 'dog_2': 1, 'weather': 2, 'code': 3}
Pairwise cosine similarity:
  Pair                            Score
  ------------------------------  -----
 dog_1 <--> dog_2               0.5934
 dog_1 <--> weather             0.2008
 dog_1 <--> code                0.0319
 dog_2 <--> weather             0.2120
None
